In [ ]:
from snowflake.snowpark.context import get_active_session

from snowflake.snowpark.functions import (
    col, trim, upper, initcap, lit, to_date, when, is_null,
    round as sf_round, sum as sf_sum, count as sf_count,
    year, month, dayofmonth, dayofweek, dayname, monthname,
    quarter, date_part, iff, coalesce
)

from snowflake.snowpark.types import DecimalType

session = get_active_session()

In [ ]:
session.sql("CREATE SCHEMA IF NOT EXISTS HOTEL_DATABASE.DEV").collect()

In [ ]:
session.sql("""
SELECT
    CURRENT_DATABASE(),
    CURRENT_SCHEMA(),
    CURRENT_ROLE()
""").show()

In [ ]:
session.sql("""
CREATE OR REPLACE STORAGE INTEGRATION S3_INT
TYPE = EXTERNAL_STAGE
STORAGE_PROVIDER = S3
ENABLED = TRUE
STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::052058222620:role/fullaccess'
STORAGE_ALLOWED_LOCATIONS = (
    's3://skanalytics-raw-data/hotel/'
);
""").collect()
#STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::052058222620:role/SnowflakeS3Role'

In [ ]:
session.sql("""
DESC INTEGRATION S3_INT;
""").show()

In [ ]:
session.sql("""
SHOW INTEGRATIONS;
""").show()

In [ ]:
session.sql("""
CREATE OR REPLACE FILE FORMAT HOTEL_DATABASE.DEV.HOTEL_CSV_FORMAT
TYPE = CSV
SKIP_HEADER = 1
FIELD_DELIMITER = ','
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('NULL','');
""").collect()

In [ ]:
session.sql("""
CREATE OR REPLACE STAGE HOTEL_DATABASE.DEV.HOTEL_STAGE
URL = 's3://skanalytics-raw-data/hotel/'
STORAGE_INTEGRATION = S3_INT
FILE_FORMAT = HOTEL_DATABASE.DEV.HOTEL_CSV_FORMAT;
""").collect()

In [ ]:
session.sql("""
SHOW STAGES IN SCHEMA HOTEL_DATABASE.DEV
""").show()

In [ ]:
session.sql("""DESC INTEGRATION S3_INT;""").collect()

In [ ]:
session.sql("""
    LIST @HOTEL_DATABASE.DEV.HOTEL_STAGE;
    """).collect()

Bronze layer

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2 (
    BOOKING_ID        VARCHAR,
    HOTEL_ID          VARCHAR,
    HOTEL_CITY        VARCHAR,
    CUSTOMER_ID       VARCHAR,
    CUSTOMER_NAME     VARCHAR,
    CUSTOMER_EMAIL    VARCHAR,
    CHECK_IN_DATE     VARCHAR,
    CHECK_OUT_DATE    VARCHAR,
    ROOM_TYPE         VARCHAR,
    NUM_GUESTS        NUMBER(10,0),
    TOTAL_AMOUNT      DECIMAL(18,2),
    CURRENCY          VARCHAR,
    BOOKING_STATUS    VARCHAR,
    SOURCE_FILE_NAME  VARCHAR,
    LOAD_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
""").collect()

In [ ]:
session.sql("""
SELECT COUNT(*) FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2
        """).collect()

Load Data into Bronze Table

In [ ]:
session.sql("""
CREATE OR REPLACE PIPE HOTEL_DATABASE.DEV.HOTEL_BOOKING_PIPE_v2
AUTO_INGEST = TRUE
AS
COPY INTO HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2
(
    BOOKING_ID,
    HOTEL_ID,
    HOTEL_CITY,
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CUSTOMER_EMAIL,
    CHECK_IN_DATE,
    CHECK_OUT_DATE,
    ROOM_TYPE,
    NUM_GUESTS,
    TOTAL_AMOUNT,
    CURRENCY,
    BOOKING_STATUS,
    SOURCE_FILE_NAME,
    LOAD_TIMESTAMP
)
FROM (
    SELECT
        $1,$2,$3,$4,$5,
        $6,$7,$8,$9,$10,
        $11,$12,$13,
        METADATA$FILENAME,
        CURRENT_TIMESTAMP()
    FROM @HOTEL_DATABASE.DEV.HOTEL_STAGE
);
""").collect()

In [ ]:
session.sql("""
    ALTER PIPE HOTEL_DATABASE.DEV.HOTEL_BOOKING_PIPE_v2 REFRESH;
    """).collect()

In [ ]:
session.sql("""
SELECT COUNT(*)
FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2;
""").collect()

In [ ]:
session.sql("""
    SELECT SYSTEM$PIPE_STATUS(
'HOTEL_DATABASE.DEV.HOTEL_BOOKING_PIPE_v2'
);
""").collect()

# ***Silver Layer*** # 

## ****Vaidate data**** ##

In [ ]:
session.sql("""
    SELECT customer_email
            FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2
            WHERE NOT (customer_email LIKE '%@%.%')
            OR customer_email IS NULL;"""
).collect()

In [ ]:
session.sql("""
SELECT TOTAL_AMOUNT
FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2
WHERE TOTAL_AMOUNT < 0;
""").collect()

In [ ]:
session.sql("""
SELECT check_in_date, check_out_date
FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2
WHERE TRY_TO_DATE(check_out_date) < TRY_TO_DATE(check_in_date);
""").collect()


In [ ]:
session.sql("""
    SELECT DISTINCT booking_status
    FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_v2;
""").collect()

# *Create Silver Table* #

In [ ]:
session.sql("""
Create table HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER AS 
  SELECT
    booking_id,
    hotel_id,
    INITCAP(TRIM(hotel_city)) AS hotel_city,
    customer_id,
    INITCAP(TRIM(customer_name)) AS customer_name,
    CASE
        WHEN customer_email LIKE '%@%.%'
            THEN LOWER(TRIM(customer_email))
        ELSE NULL
    END AS customer_email,
    check_in_date,
    check_out_date,
    room_type,
    num_guests,
    ABS(total_amount) AS total_amount,
    currency,
    CASE
        WHEN LOWER(booking_status) IN ('confirmeeed', 'confirmd')
            THEN 'Confirmed'
        ELSE booking_status
    END AS booking_status
FROM HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_V2
WHERE check_in_date IS NOT NULL
  AND check_out_date IS NOT NULL
  AND check_out_date >= check_in_date;
  """).collect()

In [ ]:
session.sql("""
    SELECT * FROM  HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER LIMIT 10;
    """).collect()

# DROP This table to create the silver from dataframe

In [ ]:
# DROP This table to create the silver from dataframe
session.sql("""DROP TABLE HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER""").collect()

In [ ]:
SRC = "HOTEL_DATABASE.DEV"

### ***DataFrame - Silver*** ###

In [ ]:
from snowflake.snowpark.functions import (
    col, trim, initcap, lower, when,
    abs as abs_, lit
)

# Read Bronze table
df_bronze = session.table("HOTEL_DATABASE.DEV.HOTEL_BOOKING_BRONZE_V2")

# Apply transformations
df_silver = (
    df_bronze
    .select(
        col("BOOKING_ID"),
        col("HOTEL_ID"),
        initcap(trim(col("HOTEL_CITY"))).alias("HOTEL_CITY"),
        col("CUSTOMER_ID"),
        initcap(trim(col("CUSTOMER_NAME"))).alias("CUSTOMER_NAME"),
        when(
            col("CUSTOMER_EMAIL").like("%@%.%"),
            lower(trim(col("CUSTOMER_EMAIL")))
        ).otherwise(None).alias("CUSTOMER_EMAIL"),
        col("CHECK_IN_DATE"),
        col("CHECK_OUT_DATE"),
        col("ROOM_TYPE"),
        col("NUM_GUESTS"),
        abs_(col("TOTAL_AMOUNT")).alias("TOTAL_AMOUNT"),
        col("CURRENCY"),
        when(
            lower(col("BOOKING_STATUS")).isin(
                "confirmeeed",
                "confirmd"
            ),
            lit("Confirmed")
        ).otherwise(col("BOOKING_STATUS")).alias("BOOKING_STATUS")
    )
    .filter(
        (col("CHECK_IN_DATE").is_not_null()) &
        (col("CHECK_OUT_DATE").is_not_null()) &
        (col("CHECK_OUT_DATE") >= col("CHECK_IN_DATE"))
    )
)

# Save to Silver table
df_silver.write.mode("overwrite").save_as_table(
    "HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER"
)

print("Silver table created successfully")

In [ ]:
df_silver

In [ ]:
df_silver.show(10)

In [ ]:
df_silver.to_pandas().head()

In [ ]:
# Row count
df_silver.count()

In [ ]:
# Column names
df_silver.columns

In [ ]:
# Schema
df_silver.print_schema()

In [ ]:
# First row
df_silver.first()

In [ ]:
# Top 5 rows
df_silver.limit(5).collect()

#### ***1. GOLD_AGG_DAILY_BOOKING*** ###

In [ ]:
from snowflake.snowpark.functions import col, count, sum as sf_sum

df_silver = session.table("HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER")

gold_daily_booking = (
    df_silver
    .group_by(col("CHECK_IN_DATE"))
    .agg(
        count("*").alias("TOTAL_BOOKING"),
        sf_sum(col("TOTAL_AMOUNT")).alias("TOTAL_REVENUE")
    )
    .sort(col("CHECK_IN_DATE"))
)

gold_daily_booking.write.mode("overwrite").save_as_table(
    "HOTEL_DATABASE.DEV.GOLD_AGG_DAILY_BOOKING"
)
print("GOLD_AGG_DAILY_BOOKING table created successfully")

In [ ]:
session.sql("""
SHOW TABLES IN SCHEMA HOTEL_DATABASE.DEV
""").show()

### ***2. GOLD_AGG_HOTEL_CITY_SALES*** ###

In [ ]:
from snowflake.snowpark.functions import sum as sf_sum

df_silver = session.table(
    "HOTEL_DATABASE.DEV.HOTEL_BOOKING_SILVER"
)

gold_hotel_city_sales = (
    df_silver 
    .group_by(col("HOTEL_CITY"))
    .agg(
        sf_sum(col("TOTAL_AMOUNT")).alias("TOTAL_REVENUE")
    )
    .sort(col("TOTAL_REVENUE"), ascending=False)
)

gold_hotel_city_sales.write.mode("overwrite").save_as_table(
    "HOTEL_DATABASE.DEV.GOLD_AGG_HOTEL_CITY_SALES"
)
print("GOLD_AGG_HOTEL_CITY_SALES table created successfully")

In [ ]:
session.sql("""
SHOW TABLES IN SCHEMA HOTEL_DATABASE.DEV
""").show()

In [ ]:
session.sql("""
SHOW TABLES IN SCHEMA HOTEL_DATABASE.DEV
""").show()

In [ ]:
from snowflake.snowpark.functions import col

gold_booking_clean = df_silver.select(
    col("BOOKING_ID"),
    col("HOTEL_ID"),
    col("HOTEL_CITY"),
    col("CUSTOMER_ID"),
    col("CUSTOMER_NAME"),
    col("CUSTOMER_EMAIL"),
    col("CHECK_IN_DATE"),
    col("CHECK_OUT_DATE"),
    col("ROOM_TYPE"),
    col("NUM_GUESTS"),
    col("TOTAL_AMOUNT"),
    col("CURRENCY"),
    col("BOOKING_STATUS")
)

gold_booking_clean.write.mode("overwrite").save_as_table(
    "HOTEL_DATABASE.DEV.GOLD_BOOKING_CLEAN"
)

print("GOLD_BOOKING_CLEAN table created successfully")

In [ ]:
gold_booking_clean.to_pandas().head()

In [ ]:
session.table(
    "HOTEL_DATABASE.DEV.GOLD_BOOKING_CLEAN"
).show()